In [ ]:
# from datasets import load_dataset
# imdb_dataset= load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
import random
from collections import Counter
from typing import List, Tuple
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Dataset

In [ ]:
SEED = 42
EMBED_DIM = 100
WINDOW_SIZE = 2
NEGATIVE_SAMPLES = 5
BATCH_SIZE = 512
EPOCHS = 5
LR = 0.002 # learning rate
NUM_DOCUMENTS = 100 # 읽을 문서 수
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
"cpu")

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

# **문서를 단어 단위로 쪼개기**

In [ ]:
def read_corpus() -> List[List[str]]:
    imdb_dataset = load_dataset("stanfordnlp/imdb")
    files: List[str] = imdb_dataset["train"]["text"][:NUM_DOCUMENTS]
    print(f"files[0][:200] = {files[0][:200]}")
    # return 구문 상세화~
    # output = []
    # for f in files:
    #   f: str
    #   inner_1 = []
    #   for w in f.split():
    #     inner_1.append(w.lower())
    #   output.append(inner_1)
    # files = output
    return [[w.lower() for w in f.split()] for f in files]


tokens_list = read_corpus()
print(f"#(documents): {len(tokens_list)}")
print("The number of tokens in the 1st document:", len(tokens_list[0]))
print(f"imdb_corpus[0][:20] = {tokens_list[0][:20]}")

# 단일 토큰 시퀀스로 flatten
all_tokens: List[str] = [tok for doc in tokens_list for tok in doc]

files[0][:200] = I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev
#(documents): 100
The number of tokens in the 1st document: 288
imdb_corpus[0][:20] = ['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']


# **전처리**

In [ ]:
def build_vocab(tokens_list: List[List[str]], tokens: List[str]) -> Tuple[dict,dict, List[List[int]]]:
  word2idx = {}
  idx2word = {}
  i=0
  #word2idx["AAAA"] = 1
  for token in tokens:
    if token not in word2idx:
      word2idx[token] = i
      idx2word[i] = token
      i += 1
  # idx2word = {v: k for k, v in word2idx.items()}
  indexed_docs : list[list[int]] = [[word2idx[token] for token in tokens] for tokens in tokens_list]
  return word2idx, idx2word, indexed_docs

word2idx, idx2word, indexed_docs = build_vocab(tokens_list, all_tokens)
vocab_size = len(word2idx)
print(f"어휘 수: {vocab_size}")
print(indexed_docs[0][:20])

어휘 수: 6008
[0, 1, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 15, 17]


# **pair 생성**

In [27]:
def build_skipgram_pairs_docs(indexed_docs: list[list[int]], window_size: int = 2) -> list[tuple[int, int]]:
    output = []
    doc_count = 0
    for doc in indexed_docs:
        print("Processing:", doc)

        if doc_count >= 3:
            break
        for c_i, c in enumerate(doc):  # 인덱스와 단어 모두 가져옴
            left = max(0, c_i - window_size)
            right = min(c_i + window_size, len(doc) - 1)
            for j in range(left, right + 1):
                if j != c_i:  # 자기 자신 제외
                    output.append((c, doc[j]))
        doc_count += 1
    return output

pairs = build_skipgram_pairs_docs(indexed_docs, window_size=1)
print(pairs[:10])

Processing: [0, 1, 0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 15, 17, 18, 19, 20, 21, 0, 22, 23, 13, 24, 18, 15, 17, 25, 26, 27, 28, 29, 15, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 9, 40, 41, 42, 0, 43, 44, 32, 45, 34, 46, 47, 48, 49, 50, 51, 52, 53, 38, 54, 55, 56, 57, 58, 59, 60, 61, 32, 62, 63, 64, 65, 66, 67, 20, 68, 64, 61, 32, 69, 70, 71, 32, 72, 73, 74, 9, 75, 76, 77, 11, 78, 79, 80, 66, 81, 82, 83, 84, 85, 11, 86, 87, 88, 89, 83, 20, 11, 90, 91, 20, 92, 93, 94, 88, 95, 96, 9, 97, 66, 98, 99, 76, 100, 64, 101, 102, 103, 70, 56, 104, 105, 88, 106, 107, 48, 108, 109, 110, 66, 0, 2, 3, 51, 13, 111, 112, 113, 34, 17, 41, 114, 115, 11, 102, 88, 116, 117, 118, 119, 88, 120, 121, 122, 123, 124, 125, 126, 127, 73, 128, 129, 130, 131, 5, 132, 133, 134, 15, 135, 20, 136, 102, 88, 116, 118, 38, 137, 138, 20, 55, 139, 122, 140, 141, 142, 98, 143, 32, 144, 145, 146, 147, 148, 44, 102, 117, 20, 149, 150, 48, 151, 152, 153, 11, 154, 46, 11, 155, 13, 156, 102, 157, 20, 11, 158, 51, 

In [ ]:
def make_unigram_probs(indexed_docs ,vocab_size: int, power: float= 0.75) -> np.ndarray:

    freqs = np.zeros(vocab_size, dtype=np.float64)
    for doc in indexed_docs:
        for i in doc:
            freqs[i] += 1
    freqs = np.power(freqs, power)
    probs = freqs**power
    probs/= probs.sum()
    return probs

probs= make_unigram_probs(indexed_docs, vocab_size)